# 01. Исследовательский анализ данных

Цель ноутбука — проверить качество ежемесячных снимков клиентов Santander, описать демографию и текущие продуктовые портфели, а затем исследовать **новые покупки**. Целевая переменная определяется только как переход продукта из `0` в месяце $t$ в `1` в соседнем месяце $t+1$.

`train_ver2.csv` загружается целиком одним вызовом `pandas.read_csv`; в анализ входят все строки и все клиенты. Chunking и выборка не применяются, поэтому перед выполнением нужна машина с достаточным объёмом RAM. Outputs очищены после изменения загрузчика: выполните все ячейки заново, чтобы получить full-data результаты.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    candidate = PROJECT_ROOT.parent
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
    else:
        raise RuntimeError("Запустите ноутбук из корня проекта или каталога notebooks")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from bank_recommender.constants import (
    DATE_COLUMN,
    ID_COLUMN,
    PRODUCT_COLUMNS,
    PRODUCT_DESCRIPTIONS_RU,
)
from bank_recommender.data import build_temporal_pairs, read_snapshots

raw_data_path = Path(os.getenv("DATA_PATH", "train_ver2.csv")).expanduser()
DATA_PATH = raw_data_path if raw_data_path.is_absolute() else PROJECT_ROOT / raw_data_path
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Датасет не найден: {DATA_PATH}")

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Данные: {DATA_PATH}")
print("Режим загрузки: один pandas.read_csv, все клиенты")

## 1. Полный набор данных и стратегия чтения

В `train_ver2.csv` фактически **13 647 309 строк данных и 48 колонок** (без строки заголовка). Первые 24 колонки описывают дату и профиль клиента, следующие 24 — наличие банковских продуктов.

`read_snapshots(DATA_PATH)` вызывает `pandas.read_csv` один раз, после чего нормализует используемые проектом поля. Все строки и клиенты обрабатываются вместе; построчная или клиентская выборка и `chunksize` отсутствуют.

In [ ]:
FULL_DATA_ROWS = 13_647_309
FULL_DATA_COLUMNS = 48
header = pd.read_csv(DATA_PATH, nrows=0)
if len(header.columns) != FULL_DATA_COLUMNS:
    raise ValueError(
        f"Ожидалось {FULL_DATA_COLUMNS} колонок, найдено {len(header.columns)}"
    )

full_dataset_facts = pd.DataFrame(
    {
        "значение": [
            FULL_DATA_ROWS,
            len(header.columns),
            len(PRODUCT_COLUMNS),
            DATA_PATH.stat().st_size / 1024**3,
        ]
    },
    index=["строк данных", "колонок", "продуктовых колонок", "размер, GiB"],
)
display(full_dataset_facts)

## 2. Стабильная выборка снимков

Загрузчик нормализует пробелы и типы, преобразует дату, приводит продуктовые индикаторы к `0/1`, сортирует снимки по клиенту и дате и оставляет последнюю запись для дубликата `(ncodpers, fecha_dato)`. Все следующие расчёты используют один объект `snapshots`, повторного сканирования CSV нет.

In [ ]:
snapshots = read_snapshots(DATA_PATH)

dataset_overview = pd.DataFrame(
    {
        "значение": [
            snapshots.shape[0],
            snapshots.shape[1],
            snapshots[ID_COLUMN].nunique(),
            snapshots[DATE_COLUMN].nunique(),
            snapshots[DATE_COLUMN].min().date(),
            snapshots[DATE_COLUMN].max().date(),
        ]
    },
    index=[
        "строк после нормализации",
        "колонок после нормализации",
        "уникальных клиентов",
        "месячных срезов",
        "минимальная дата",
        "максимальная дата",
    ],
)
display(dataset_overview)
display(snapshots.head(3))

## 3. Качество данных: дубликаты и пропуски

Дубликаты ниже проверяются после штатной нормализации: ожидается ноль повторов ключа клиент–месяц. Пропуски в профиле сохраняются и будут обрабатываться внутри ML-pipeline. Пропуски продуктовых индикаторов трактуются загрузчиком как отсутствие продукта (`0`); это решение необходимо явно учитывать при интерпретации редких ранних записей.

In [ ]:
duplicate_rows = int(snapshots.duplicated().sum())
duplicate_customer_months = int(
    snapshots.duplicated([ID_COLUMN, DATE_COLUMN]).sum()
)
quality_checks = pd.Series(
    {
        "полных дубликатов после нормализации": duplicate_rows,
        "дубликатов клиент-месяц после нормализации": duplicate_customer_months,
        "строк с пропуском идентификатора": int(snapshots[ID_COLUMN].isna().sum()),
        "строк с пропуском даты": int(snapshots[DATE_COLUMN].isna().sum()),
    },
    name="количество",
)
missing_summary = (
    pd.DataFrame(
        {
            "missing_count": snapshots.isna().sum(),
            "missing_pct": snapshots.isna().mean().mul(100),
        }
    )
    .sort_values(["missing_pct", "missing_count"], ascending=False)
)

display(quality_checks.to_frame())
display(missing_summary.head(15))

In [ ]:
missing_plot = (
    missing_summary.head(12)
    .sort_values("missing_pct")
    .reset_index(names="column")
)
plt.figure(figsize=(9, 5))
sns.barplot(data=missing_plot, x="missing_pct", y="column", color="#4C72B0")
plt.title("Пропуски в полном наборе данных")
plt.xlabel("Доля пропусков, %")
plt.ylabel("Признак")
plt.tight_layout()
plt.show()

## 4. Демографический профиль

Возраст и доход анализируются только после числового приведения. Для категориальных полей выводятся несколько наиболее частых значений, чтобы не создавать таблицы на сотни категорий. Экстремальные и отсутствующие значения нельзя заполнять по всему датасету до временного split: медианы и словари категорий должны обучаться только на train.

In [ ]:
numeric_profile = snapshots[["age", "antiguedad", "renta"]].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T
category_rows = []
for column in ["sexo", "segmento", "ind_actividad_cliente", "pais_residencia"]:
    shares = snapshots[column].astype("string").fillna("<NA>").value_counts(
        normalize=True
    ).head(5)
    category_rows.extend(
        {"признак": column, "значение": value, "доля": share}
        for value, share in shares.items()
    )

display(numeric_profile)
display(pd.DataFrame(category_rows))

In [ ]:
valid_age = pd.to_numeric(snapshots["age"], errors="coerce")
valid_age = valid_age[valid_age.between(18, 100)]
plt.figure(figsize=(9, 4.5))
sns.histplot(valid_age, bins=40, color="#55A868")
plt.title("Распределение возраста после ограничения 18–100 лет")
plt.xlabel("Возраст")
plt.ylabel("Количество снимков")
plt.tight_layout()
plt.show()

## 5. Текущий продуктовый портфель

Индикаторы в снимке отвечают на вопрос «какие продукты уже есть у клиента», но это ещё не target. Их частоты важны как baseline и как признаки текущего портфеля. Сильная концентрация владений вокруг нескольких массовых продуктов означает, что обычная accuracy будет почти полностью определяться отрицательным классом.

In [ ]:
current_product_table = pd.DataFrame(
    {
        "product": PRODUCT_COLUMNS,
        "description": [PRODUCT_DESCRIPTIONS_RU[p] for p in PRODUCT_COLUMNS],
        "owners_in_dataset": [int(snapshots[p].sum()) for p in PRODUCT_COLUMNS],
        "snapshot_prevalence": [float(snapshots[p].mean()) for p in PRODUCT_COLUMNS],
    }
).sort_values("snapshot_prevalence", ascending=False, ignore_index=True)
display(current_product_table.head(12))

In [ ]:
product_plot = current_product_table.head(12).sort_values("snapshot_prevalence")
plt.figure(figsize=(10, 6))
sns.barplot(
    data=product_plot,
    x="snapshot_prevalence",
    y="description",
    color="#C44E52",
)
plt.title("Наиболее распространённые текущие продукты")
plt.xlabel("Доля клиентских снимков")
plt.ylabel("Продукт")
plt.tight_layout()
plt.show()

## 6. Новые покупки: переходы 0→1

`build_temporal_pairs` соединяет клиента только с **соседним календарным месяцем**. Для каждой продуктовой колонки вычисляется $y=\max(x_{t+1}-x_t, 0)$. Снятие продукта (`1→0`) не является покупкой, а разрыв в несколько месяцев не создаёт пару. Признаки остаются в месяце $t$, поэтому будущая информация не попадает в модель.

In [ ]:
pairs = build_temporal_pairs(snapshots)
additions_per_pair = pairs.targets[PRODUCT_COLUMNS].sum(axis=1)
target_month = (pairs.periods + 1).astype(str)

pair_summary = pd.Series(
    {
        "соседних пар клиент-месяц": len(pairs.features),
        "всего событий 0→1": int(additions_per_pair.sum()),
        "доля пар хотя бы с одной покупкой": float((additions_per_pair > 0).mean()),
        "среднее число покупок на пару": float(additions_per_pair.mean()),
        "максимум покупок в одной паре": int(additions_per_pair.max()),
    },
    name="значение",
)
addition_table = pd.DataFrame(
    {
        "product": PRODUCT_COLUMNS,
        "description": [PRODUCT_DESCRIPTIONS_RU[p] for p in PRODUCT_COLUMNS],
        "addition_events": [int(pairs.targets[p].sum()) for p in PRODUCT_COLUMNS],
        "addition_rate": [float(pairs.targets[p].mean()) for p in PRODUCT_COLUMNS],
    }
).sort_values("addition_events", ascending=False, ignore_index=True)

display(pair_summary.to_frame())
display(addition_table.head(12))

In [ ]:
target_frame = pairs.targets[PRODUCT_COLUMNS].copy()
target_frame["target_month"] = target_month.to_numpy()
monthly_product_additions = target_frame.groupby("target_month")[PRODUCT_COLUMNS].sum()
monthly_pair_count = target_frame.groupby("target_month").size()
monthly_additions = pd.DataFrame(
    {
        "addition_events": monthly_product_additions.sum(axis=1),
        "customer_pairs": monthly_pair_count,
    }
)
monthly_additions["events_per_1000_pairs"] = (
    1000 * monthly_additions["addition_events"] / monthly_additions["customer_pairs"]
)
display(monthly_additions)

plt.figure(figsize=(10, 4.5))
sns.lineplot(
    data=monthly_additions.reset_index(),
    x="target_month",
    y="events_per_1000_pairs",
    marker="o",
    color="#8172B2",
)
plt.title("Динамика новых покупок по целевому месяцу")
plt.xlabel("Месяц покупки (t+1)")
plt.ylabel("Событий 0→1 на 1000 пар")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Выводы и последствия для моделирования

1. **Цель существенно разрежена.** Большинство пар клиент–месяц не содержит ни одной покупки, а редкие продукты дают крайне мало положительных примеров. Accuracy и ROC-AUC могут выглядеть высокими при бесполезном ранжировании, поэтому основная offline-метрика проекта — MAP@7, дополненная Recall@7, Precision@7, coverage и macro PR-AUC.
2. **Текущие владения и новые покупки — разные величины.** Популярность текущих счетов нельзя выдавать за частоту target. Обучение использует только переходы `0→1`; продукты, уже имеющиеся у клиента в $t$, должны маскироваться при ранжировании.
3. **Пропуски неоднородны.** Профильные поля содержат отсутствующие значения, а некоторые категории редки. Заполнение числовых признаков, кодирование категорий и обработка неизвестных значений должны находиться внутри pipeline, обученного только на train. Продуктовые пропуски загрузчик приводит к нулю как документированное допущение.
4. **Нужен временной split.** Случайное разделение строк одного клиента смешало бы прошлое и будущее. Для финальной оценки признаки апреля 2016 года используются для прогноза покупок мая; все более ранние пары образуют train.
5. **Все клиенты обрабатываются вместе.** CSV загружается одним чтением без chunking и sampling. Это исключает расхождение между демонстрационной выборкой и полным набором, но требует существенно больше RAM и времени вычисления.
6. **Прогноз после доступной истории.** Последний снимок относится к маю 2016 года, поэтому после выбора конфигурации модель можно переобучить на всех наблюдаемых соседних парах и применять к майскому портфелю для ранжирования продуктов июня.